# Blazegraph Database

[Blazegraph](https://blazegraph.com/) is an open-source RDF triplestore (graph database) with a SPARQL 1.1 endpoint. It is the model database of the [GridAPPS-D](3_7_gridappsd.ipynb) platform.

CIMantic Graphs provides the `BlazegraphConnection` class for reading and updating CIM models in any Blazegraph SPARQL endpoint. It is a `SPARQLEndpointConnection`, so it shares its query and parsing methods with [GraphDB](3_5_graphdb.ipynb).

## Overview

**Key Features:**
* SPARQL 1.1 queries and updates over HTTP
* Handles large distribution models such as the IEEE 9500-node feeder
* Fast bulk loading of RDF/XML files
* Built-in web workbench for running SPARQL queries
* Docker images, including GridAPPS-D images with the IEEE test feeders loaded

**Best For:**
* Large distribution feeder models
* Applications using the GridAPPS-D platform
* Development and testing with the IEEE test feeders

----

## Installation and Setup

### Python Dependencies

`BlazegraphConnection` needs the `SPARQLWrapper` library, from the `sparql` extra. `pip install cim-graph` alone does not install it:

```bash
pip install "cim-graph[sparql]"
```

Without it, `from cimgraph.databases import BlazegraphConnection` raises an `ImportError` naming the extra to install.

### Local Instance (docker compose)

The repository's `docker-compose.yml` runs Blazegraph 2.1.5 for the integration tests. From a CIM-Graph checkout:

```bash
docker compose up -d --wait blazegraph
uv run python tests/seed_databases.py blazegraph   # loads tests/test_models/ieee13.xml
```

The SPARQL endpoint is `http://localhost:8889/bigdata/namespace/kb/sparql`. The container has no volume, so `docker compose down` discards the data. The examples on this page use this instance.

### GridAPPS-D Docker Images

GridAPPS-D publishes Blazegraph images with the IEEE test feeders already loaded. They also use port 8889, so stop the docker compose `blazegraph` service first.

**For CIM100 / cimhub_2023 profile (GridAPPS-D v2025.01.0 and later):**
```bash
docker run -p 8889:8080 gridappsd/blazegraph:v2025.01.0
```

**For CIM17 / rc4_2021 profile (GridAPPS-D v2024.09.0 and earlier):**
```bash
docker run -p 8889:8080 gridappsd/blazegraph:v2024.09.0
```

Available tags: https://hub.docker.com/r/gridappsd/blazegraph/tags

### Standalone Installation

1. Download the Blazegraph JAR from https://github.com/blazegraph/database/releases
2. Start the server:
   ```bash
   java -server -Xmx4g -jar blazegraph.jar
   ```
3. Open the web interface at http://localhost:9999/blazegraph/

----

## Environment Configuration

A connection reads its settings from `CIMG_*` environment variables when it is created (see [Environment Variables](3_2_env_variables.ipynb)). Blazegraph uses three of them:

| Deployment | `CIMG_CIM_PROFILE` | `CIMG_IEC61970_552` | `CIMG_URL` |
|---|---|---|---|
| docker compose, GridAPPS-D v2025.01.0+ | `cimhub_2023` | `552-NEW` | `http://localhost:8889/bigdata/namespace/kb/sparql` |
| GridAPPS-D v2024.09.0 and earlier | `rc4_2021` | `552-LEGACY` | `http://localhost:8889/bigdata/namespace/kb/sparql` |
| Standalone JAR | the profile of your data | the identifier style of your data | `http://localhost:9999/blazegraph/namespace/kb/sparql` |

Import the matching profile for the `cim` classes: `cimprofiles.cim17.cimhub_2023` or `cimprofiles.cim17.gridappsd_rc4_2021`. For the docker compose instance:

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
os.environ['CIMG_URL'] = 'http://localhost:8889/bigdata/namespace/kb/sparql'
os.environ['CIMG_IEC61970_552'] = '552-NEW'

import cimprofiles.cim17.cimhub_2023 as cim

----

## Creating a Connection

Import and instantiate the `BlazegraphConnection` class:

In [2]:
from cimgraph.databases import BlazegraphConnection

db = BlazegraphConnection()

Creating the connection reads the `CIMG_*` settings and builds the SPARQLWrapper client. Nothing is sent to Blazegraph until the first query. Later changes to the environment variables do not affect an existing connection: create a new one.

----

## Blazegraph Implementation

`BlazegraphConnection` (`cimgraph/databases/blazegraph/blazegraph.py`) extends `SPARQLEndpointConnection` by implementing its database-specific methods with [SPARQLWrapper](https://sparqlwrapper.readthedocs.io/):

| Method | Implementation |
|---|---|
| `__init__()` | Reads the `CIMG_*` settings, takes the endpoint URL from `CIMG_URL` and calls `connect()` |
| `_setup_connection()` | Creates the `SPARQLWrapper` client used for updates, with JSON results |
| `_execute_raw_query(query_message)` | Sends the query by POST with a new `SPARQLWrapper` and returns the SPARQL JSON results as a `dict`. Each call has its own client, so batch queries can run in parallel threads |
| `_parse_result_field(result, field_name)` | Returns `result[field_name]['value']`, or `None` if the field is unbound |
| `_update_raw(update_message)` | Sends a SPARQL Update by POST to the same endpoint |

----

## Common SPARQL Endpoint Methods

`BlazegraphConnection` inherits all standard methods from `SPARQLEndpointConnection`. They are documented in the [Databases Overview](3_1_databases_overview.ipynb#common-sparql-endpoint-methods).

**Inherited Methods:**
* **Connection Management:** `connect()`, `disconnect()`
* **Query Execution:** `execute()`, `update()`
* **Object Retrieval:** `get_object()`, `get_from_triple()`
* **Graph Creation:** `create_new_graph()`, `create_distributed_graph()`, `build_graph_from_list()`
* **Graph Expansion:** `get_all_edges()`, `get_all_attributes()`, `get_edges_query()`
* **Query Parsing:** `parse_node_query()`, `edge_query_parser()`
* **Data Upload:** `upload()`

----

## Usage Examples

### Example 1: Loading a Feeder Model

In [3]:
from cimgraph.models import FeederModel

# Load the IEEE 13-bus feeder
feeder = cim.Feeder(mRID="49AD8E07-3BF9-A4E2-CB8F-C3722F837B62")
network = FeederModel(container=feeder, connection=db)

print(f"Loaded {len(network.graph[cim.ACLineSegment])} line segments")

Loaded 11 line segments


### Example 2: Retrieving an Object by mRID

In [4]:
feeder = db.get_object(mRID="49AD8E07-3BF9-A4E2-CB8F-C3722F837B62")
feeder.pprint()

{
    "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
    "@type": "Feeder"
}


### Example 3: Querying Object Attributes

In [5]:
# A string attribute
name = db.get_from_triple(subject=feeder, predicate='IdentifiedObject.name')
print(f"Feeder name: {name}")

# An association: a list of CIM objects
substation = db.get_from_triple(subject=feeder, predicate='Feeder.NormalEnergizingSubstation')
print(f"Substation: {substation}")

Feeder name: ['ieee13nodeckt']
Substation: [{"@id": "6c62c905-6fc7-653d-9f1e-1340f974a587", "@type": "Substation"}]


### Example 4: Executing a Custom SPARQL Query

`execute()` returns the SPARQL JSON results as a `dict`:

In [6]:
query_text = '''
PREFIX r:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX cim:  <http://iec.ch/TC57/CIM100#>
SELECT DISTINCT ?identifier ?name
WHERE {
    ?feeder r:type cim:Feeder .
    BIND(strafter(str(?feeder),"urn:uuid:") as ?identifier)
    OPTIONAL { ?feeder cim:IdentifiedObject.name ?name . }
}
ORDER BY ?name
'''

results = db.execute(query_text)

print("Available Feeders:")
for result in results['results']['bindings']:
    mrid = result['identifier']['value']
    name = result.get('name', {}).get('value', 'Unnamed')
    print(f"  - {name}: {mrid}")

Available Feeders:
  - ieee13nodeckt: 49ad8e07-3bf9-a4e2-cb8f-c3722f837b62


### Example 5: Expanding the Graph

`get_all_edges()` adds the attributes and associated objects of every object of a class. With `CIMG_USE_UNITS` unset or `true`, unit values are `CIMUnit` objects in SI base units; convert them with `.to()`:

In [7]:
network.get_all_edges(cim.Breaker)
network.get_all_edges(cim.ACLineSegment)

breaker = network.first(cim.Breaker)
print(f"Breaker {breaker.name}: rated current {breaker.ratedCurrent.to('A')} A")

line = network.first(cim.ACLineSegment)
print(f"Line {line.name}: length {line.length.to('m')} m ({line.length.to('ft'):.0f} ft)")

Breaker brkr1: rated current 400.0 A
Line 645646: length 91.44 m (300 ft)


### Example 6: Debugging SPARQL Queries

In [8]:
# The SPARQL query that get_all_edges(cim.Breaker) runs (first batch of 100 objects)
query_text = db.get_edges_query(network.graph, cim.Breaker)
print(query_text)


        PREFIX r:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX cim:  <http://iec.ch/TC57/CIM100#>
        SELECT DISTINCT ?identifier ?attribute ?value ?edge
        WHERE {
          
    VALUES ?identifier { "52de9189-20dc-4c73-bdee-e960fe1f9493" 
               }
        bind(iri(concat("urn:uuid:", ?identifier)) as ?eq)

        ?eq r:type cim:Breaker.

        {?eq (cim:|!cim:) ?val.
         ?eq ?attr ?val.}
        UNION
        {?val (cim:|!cim:) ?eq.
         ?val ?attr ?eq.}

        {bind(strafter(str(?attr),"#") as ?attribute)}
        {bind(strafter(str(?val),"urn:uuid:") as ?uri)}
        {bind(if(?uri = "", ?val, ?uri) as ?value)}

        OPTIONAL {?val a ?classraw.
                  bind(strafter(str(?classraw),"#") as ?edge_class)
                  {bind(strafter(str(?val),"urn:uuid:") as ?uri)}

                  bind(concat("{\"@id\":\"", ?uri,"\",\"@type\":\"", ?edge_class, "\"}") as ?edge)}
        }

        ORDER by  ?identifier ?attribute
    

Copy the query into the Blazegraph web interface to run and debug it.

----

## About Triplestore Databases

The triple-store database offers a semantic solution for data management. Unlike relational databases (which require a DDL database schema), the triple-store database structure is comprised of Resource Description Framework (RDF) statements.

### RDF Triple Structure

RDF statements take the form of **subject (node) - predicate (relation) - object (node)** that can be dynamically generated to form inter-related complex class structures.

**Example CIM Triple:**
```
Subject:   ACLineSegment (urn:uuid:1234...)
Predicate: ACLineSegment.length
Object:    105 meters
```

### Advantages for CIM

The RDF statement structure intuitively corresponds to the structure of object-attribute specifications used in CIM. Key advantages include:

* **Direct CIM Support:** CIM models translate directly to RDF with automated data correlation
* **Polymorphism:** RDF Schema (RDFS) supports inheritance and class hierarchies
* **Graph Constructs:** Native support for complex graph structures and subgraphs
* **Standards-Based:** Uses mature, standardized languages (RDF, RDFS, OWL, SPARQL)
* **Validation:** Supports type-checking and SHACL (Shapes Constraint Language)
* **Agility:** Dynamic data structures support multiple developers and evolving schemas

### Serialization Formats

RDF can be serialized in multiple formats:
* **RDF/XML** - CIM XML (IEC 61970-552)
* **Turtle (TTL)** - Human-readable format
* **JSON-LD** - JSON-based RDF for web applications
* **N-Triples** - Simple line-based format

### Data Management Considerations

**Advantages:**
* No schema migration required for model changes
* Easy integration of multiple data sources
* Standardized query language (SPARQL)
* Support for reasoning and inference (with OWL)

**Challenges:**
* Risk of "garbage-in-garbage-out" without data governance
* Potential for dangling references without validation
* Requires rigor from data contributors
* Need for well-thought-out data management strategy

----

## UML Sequence Diagrams

These sequence diagrams show how `BlazegraphConnection` runs database queries and API calls. They are written as mermaid.js text.

### Blazegraph Connection Initialization

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant BlazegraphConnection
    participant SPARQLWrapper

    note right of User: Initialization

    User ->>+ BlazegraphConnection: BlazegraphConnection()
    BlazegraphConnection ->> BlazegraphConnection: read CIMG_* settings
    BlazegraphConnection ->> BlazegraphConnection: connect()
    alt connection_obj is None
        BlazegraphConnection ->> SPARQLWrapper: create SPARQLWrapper(url)
        BlazegraphConnection ->> SPARQLWrapper: setReturnFormat(JSON)
    end
    BlazegraphConnection -->>- User: connection object
```

### get_object() - Retrieving Object by mRID

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant BlazegraphConnection
    participant SPARQL Queries
    participant Database

    note right of User: Requests object of given mRID

    User ->>+ BlazegraphConnection: get_object(mRID)
    BlazegraphConnection ->>+ SPARQL Queries: get_object_sparql(mRID)
    SPARQL Queries -->>- BlazegraphConnection: str[query text]
    BlazegraphConnection ->>+ Database: execute(query_text)
    Database -->>- BlazegraphConnection: json[query response]
    BlazegraphConnection ->> BlazegraphConnection: create_object()
    BlazegraphConnection -->>- User: CIM object
```

### get_from_triple() - Querying Object Attributes

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant BlazegraphConnection
    participant SPARQL Queries
    participant Database
    note right of User: Query using RDF triple
    User ->>+ BlazegraphConnection: get_from_triple(subject, predicate)
    BlazegraphConnection ->> BlazegraphConnection: add_to_graph(subject, graph)
    BlazegraphConnection ->>+ SPARQL Queries: get_triple_sparql(subject, predicate)
    SPARQL Queries -->>- BlazegraphConnection: str[query text]
    BlazegraphConnection ->>+ Database: execute(query_text)
    Database -->>- BlazegraphConnection: json[query response]
    BlazegraphConnection ->> BlazegraphConnection: edge_query_parser()
    BlazegraphConnection -->>- User: list[CIM object or value]
```

### execute() - Executing SPARQL Query

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant BlazegraphConnection
    participant SPARQLWrapper
    participant BlazegraphEndpoint
    note right of User: Executes SPARQL query
    User ->>+ BlazegraphConnection: execute(query_message)
    BlazegraphConnection ->> BlazegraphConnection: connect()
    BlazegraphConnection ->> SPARQLWrapper: new SPARQLWrapper(url), setReturnFormat(JSON)
    BlazegraphConnection ->> SPARQLWrapper: setQuery(query_message)
    BlazegraphConnection ->> SPARQLWrapper: setMethod(POST)
    BlazegraphConnection ->>+ SPARQLWrapper: query()
    SPARQLWrapper ->>+ BlazegraphEndpoint: HTTP POST request
    BlazegraphEndpoint -->>- SPARQLWrapper: HTTP response
    SPARQLWrapper ->> SPARQLWrapper: convert()
    SPARQLWrapper -->>- BlazegraphConnection: dict[SPARQL JSON results]
    BlazegraphConnection -->>- User: query_output
```

### update() - Executing SPARQL Update

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant BlazegraphConnection
    participant SPARQLWrapper
    participant BlazegraphEndpoint
    note right of User: Executes SPARQL update
    User ->>+ BlazegraphConnection: update(update_message)
    BlazegraphConnection ->> BlazegraphConnection: connect()
    BlazegraphConnection ->> SPARQLWrapper: setQuery(update_message)
    BlazegraphConnection ->> SPARQLWrapper: setMethod(POST)
    BlazegraphConnection ->>+ SPARQLWrapper: query()
    SPARQLWrapper ->>+ BlazegraphEndpoint: HTTP POST request
    BlazegraphEndpoint -->>- SPARQLWrapper: HTTP response
    SPARQLWrapper -->>- BlazegraphConnection: response
    BlazegraphConnection -->>- User: output
```

### create_new_graph() - Building Feeder Topology

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant FeederModel
    participant BlazegraphConnection
    participant SPARQL Queries
    participant Database

    note right of User: Creates new FeederModel

    User ->>+ FeederModel: FeederModel(container, connection)
    FeederModel ->>+ BlazegraphConnection: create_new_graph(container, graph)
    BlazegraphConnection ->> BlazegraphConnection: add_to_graph(container, graph)
    BlazegraphConnection ->>+ SPARQL Queries: get_all_nodes_from_container(container)
    SPARQL Queries -->>- BlazegraphConnection: str[query text]
    BlazegraphConnection ->>+ Database: execute(query_text)
    Database -->>- BlazegraphConnection: json[query response]
    BlazegraphConnection ->> BlazegraphConnection: parse_node_query(graph, query_output)
    BlazegraphConnection -->>- FeederModel: dict[type, dict[UUID, object]]
    FeederModel -->>- User: network object
```

### create_distributed_graph() - Building Distributed Model

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant DistributedArea
    participant BlazegraphConnection
    participant SPARQL Queries
    participant Database
    note right of User: Creates graph for area
    User ->>+ DistributedArea: DistributedArea(container=area, connection, distributed=True)
    DistributedArea ->>+ BlazegraphConnection: create_distributed_graph(area, graph)
    BlazegraphConnection ->> BlazegraphConnection: add_to_graph(area, graph)
    BlazegraphConnection ->> BlazegraphConnection: type check area is SubSchedulingArea
    alt area is not SubSchedulingArea
        BlazegraphConnection -->> User: TypeError
    end
    BlazegraphConnection ->>+ SPARQL Queries: get_all_nodes_from_area(area)
    SPARQL Queries -->>- BlazegraphConnection: str[query text]
    BlazegraphConnection ->>+ Database: execute(query_text)
    Database -->>- BlazegraphConnection: json[query response]
    BlazegraphConnection ->> BlazegraphConnection: parse_node_query(graph, query_output)
    BlazegraphConnection -->>- DistributedArea: dict[type, dict[UUID, object]]
    DistributedArea -->>- User: area model
```

### get_all_edges() - Parallel Graph Expansion

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant GraphModel
    participant BlazegraphConnection
    participant ThreadPoolExecutor
    participant SPARQL Queries
    participant Database
    note right of User: Expands knowledge graph

    User ->>+ GraphModel: get_all_edges(cim_class)
    GraphModel ->>+ BlazegraphConnection: get_all_edges(graph, cim_class)
    BlazegraphConnection ->> BlazegraphConnection: split graph[cim_class] keys into batches of 100
    loop For each batch
        BlazegraphConnection ->>+ SPARQL Queries: get_all_edges_sparql(graph, cim_class, batch)
        SPARQL Queries -->>- BlazegraphConnection: str[query text]
    end
    BlazegraphConnection ->>+ ThreadPoolExecutor: run_batch_queries(queries), up to 8 at once
    loop For each query, in parallel threads
        ThreadPoolExecutor ->>+ Database: _execute_raw_query(query_text)
        Database -->>- ThreadPoolExecutor: json[query response]
    end
    ThreadPoolExecutor -->>- BlazegraphConnection: query responses, in batch order
    loop For each query response, on the calling thread
        BlazegraphConnection ->> BlazegraphConnection: edge_query_parser(query_output, graph, cim_class)
    end
    BlazegraphConnection -->>- GraphModel: None
    GraphModel -->>- User: None
```

### get_edges_query() - Query Debugging

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant BlazegraphConnection
    participant SPARQL Queries
    note right of User: Requests SPARQL query for debugging
    User ->>+ BlazegraphConnection: get_edges_query(graph, cim_class)
    BlazegraphConnection ->> BlazegraphConnection: Get first 100 keys from graph[cim_class]
    BlazegraphConnection ->>+ SPARQL Queries: get_all_edges_sparql(graph, cim_class, eq_mrids)
    SPARQL Queries -->>- BlazegraphConnection: str[query text]
    BlazegraphConnection -->>- User: SPARQL query string
```

### parse_node_query() - Parsing Network Topology

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant FeederModel
    participant BlazegraphConnection
    participant SPARQL Queries
    participant Database

    note right of User: Creates new FeederModel

    User ->>+ FeederModel: FeederModel(container, connection)
    FeederModel ->>+ BlazegraphConnection: create_new_graph(container, graph)
    BlazegraphConnection ->>+ SPARQL Queries: get_all_nodes_from_container(container)
    SPARQL Queries -->>- BlazegraphConnection: str[query text]
    BlazegraphConnection ->>+ Database: execute(query_text)
    Database -->>- BlazegraphConnection: json[query response]
    note over BlazegraphConnection: Parse Node Query Results

    loop For each result in query response
        BlazegraphConnection ->> BlazegraphConnection: Parse Equipment JSON-LD
        alt equipment class in data profile
            BlazegraphConnection ->> BlazegraphConnection: create_object(Equipment)
            BlazegraphConnection ->> BlazegraphConnection: create_object(ConnectivityNode)
            BlazegraphConnection ->> BlazegraphConnection: create_object(Terminal)
            BlazegraphConnection ->> BlazegraphConnection: create_object(Measurement)
        else equipment class not in data profile
            BlazegraphConnection ->> BlazegraphConnection: log warning
        end
    end

    BlazegraphConnection -->>- FeederModel: dict[type, dict[UUID, object]]
    FeederModel -->>- User: network object
```

### edge_query_parser() - Parsing Edge Query Results

```mermaid
%%{init: {"theme":"base"}}%%
sequenceDiagram
    actor User
    participant GraphModel
    participant BlazegraphConnection
    participant SPARQL Queries
    participant Database

    note right of User: Expands knowledge graph

    User ->>+ GraphModel: get_all_edges(cim_class)
    GraphModel ->>+ BlazegraphConnection: get_all_edges(graph, cim_class)
    BlazegraphConnection ->>+ SPARQL Queries: get_all_edges_sparql(graph, cim_class, batch)
    SPARQL Queries -->>- BlazegraphConnection: str[query text]
    BlazegraphConnection ->>+ Database: execute(query_text)
    Database -->>- BlazegraphConnection: json[query response]
    note over BlazegraphConnection: Parse Edge Query Results

    loop For each result in query response
        BlazegraphConnection ->> BlazegraphConnection: Parse Graph Edges JSON-LD
        alt is edge to another object
            BlazegraphConnection ->> BlazegraphConnection: create_edge(association, edge_uri)
        else is enumeration
            BlazegraphConnection ->> BlazegraphConnection: setattr(attribute, enum_class.enum_value)
        else is value
            BlazegraphConnection ->> BlazegraphConnection: create_value(attribute, value)
        end
    end

    BlazegraphConnection -->>- GraphModel: None
    GraphModel -->>- User: None
```

----

## Blazegraph Web Interface

Blazegraph provides a built-in web interface for interactive querying and debugging.

**Access:**
* docker compose and GridAPPS-D images: http://localhost:8889/bigdata/
* Standalone: http://localhost:9999/blazegraph/

**Features:**
* SPARQL query editor with result display
* SPARQL update and data loading (RDF files)
* Namespace management
* Database status and statistics

**Usage Tip:** Copy SPARQL queries from `get_edges_query()` into the web interface to test and debug query performance.

----

## Performance Optimization

### Memory Configuration

For large models, increase the JVM heap size of a standalone server:
```bash
java -server -Xmx8g -jar blazegraph.jar
```

### Query Batching and Parallel Execution

`get_all_edges()` and `get_all_attributes()` put 100 objects in each query (`BATCH_SIZE` in `cimgraph.databases`) and run up to 8 queries at once (`MAX_PARALLEL_QUERIES`) in a `ThreadPoolExecutor`. The results are parsed into the graph on the calling thread.

### Indexing

Blazegraph automatically indexes RDF triples for fast query execution. No manual index configuration is required.

----

## Troubleshooting

### Connection Refused
```
urllib.error.URLError: <urlopen error [Errno 111] Connection refused>
```
Raised by the first query, not when the connection is created.
* Verify Blazegraph is running
* Check the `CIMG_URL` environment variable
* Ensure the correct port (8889 for docker compose and GridAPPS-D, 9999 for standalone)

### Empty Results
* Verify the data is loaded into Blazegraph
* Check that `CIMG_NAMESPACE` (default `http://iec.ch/TC57/CIM100#`) matches the CIM namespace of the data
* Check that `CIMG_IEC61970_552` matches the identifier style of the data (`urn:uuid:` or `_` identifiers)
* Use the Blazegraph web interface to verify the data exists

### Profile Mismatch
```
object class missing from data profile: XYZ
```
* Verify `CIMG_CIM_PROFILE` matches the database content
* Check that the Docker image version corresponds to the profile

----

## Loading Data into Blazegraph

### Via Web Interface

1. Navigate to http://localhost:8889/bigdata/
2. Click the "Update" tab
3. Select "File Path or URL"
4. Choose the RDF/XML format for CIM XML files
5. Click "Update" to load the data

### Via REST API

Bulk load an RDF/XML file with curl (this is what `tests/seed_databases.py` does):
```bash
curl -X POST \
  -H 'Content-Type: application/rdf+xml' \
  --data-binary @model.xml \
  http://localhost:8889/bigdata/namespace/kb/sparql
```

### Via CIMantic Graphs

`upload()` writes every object in a graph with one SPARQL `INSERT DATA` update per object. For example, to load a CIM XML file:

```python
from cimgraph.databases import BlazegraphConnection, XMLFile
from cimgraph.models import FeederModel

xml_file = XMLFile(filename='tests/test_models/ieee13.xml')
feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(container=feeder, connection=xml_file)

db = BlazegraphConnection()
db.upload(network.graph)
```

----

## References

* [Blazegraph Official Website](https://blazegraph.com/)
* [Blazegraph GitHub](https://github.com/blazegraph/database)
* [GridAPPS-D Blazegraph Docker Images](https://hub.docker.com/r/gridappsd/blazegraph/tags)
* [SPARQL 1.1 Specification](https://www.w3.org/TR/sparql11-query/)
* [RDF 1.1 Primer](https://www.w3.org/TR/rdf11-primer/)
* [Common SPARQL Methods](3_1_databases_overview.ipynb#common-sparql-endpoint-methods)
* [FeederModel Usage](../04_graph_models/4_2_feeder_model.ipynb)